# Upsolving - Parcial 1

https://docs.google.com/document/d/1oQSpFq048tCjootXKL-NnoeSYOmjaMWqLXEFF13ucuw/edit?usp=sharing

# Punto 1

In [2]:
from typing import List, Tuple, Literal, Union

Dir = Literal["ROOT", "L", "R", "U", "D"]
Resultado = Union[Tuple[str, str, str, str], str]

def explorar_strings_matriz(matriz: List[List[str]], pos: Tuple[int, int], dir: Dir = "ROOT") -> Resultado:
    i, j = pos
    filas = len(matriz)
    columnas = len(matriz[0])

    # Validación básica de rango
    if not (0 <= i < filas and 0 <= j < columnas):
        raise IndexError("La posición inicial está fuera de la matriz.")

    actual = matriz[i][j]

    if dir == "ROOT":
        # Orden: (Izquierda, Derecha, Arriba, Abajo)
        return (
            explorar_strings_matriz(matriz, pos, "L"),
            explorar_strings_matriz(matriz, pos, "R"),
            explorar_strings_matriz(matriz, pos, "U"),
            explorar_strings_matriz(matriz, pos, "D"),
        )

    if dir == "L":
        # Caso base: ya estamos en la primera columna
        if j == 0:
            return actual
        # Paso recursivo NO de cola: concatenamos tras evaluar la recursión
        return actual + explorar_strings_matriz(matriz, (i, j - 1), "L")

    if dir == "R":
        if j == columnas - 1:
            return actual
        return actual + explorar_strings_matriz(matriz, (i, j + 1), "R")

    if dir == "U":
        if i == 0:
            return actual
        return actual + explorar_strings_matriz(matriz, (i - 1, j), "U")

    if dir == "D":
        if i == filas - 1:
            return actual
        return actual + explorar_strings_matriz(matriz, (i + 1, j), "D")

    # Si llega aquí, 'dir' no fue válido
    raise ValueError("Dirección no válida. Use 'ROOT', 'L', 'R', 'U' o 'D'.")


In [5]:
matriz = [
    ["A", "B", "C", "X"],
    ["D", "E", "F", "Y"],
    ["G", "H", "I", "Z"],
    ["M", "N", "Ñ", "O"],
]
pos_inicial = (1, 1)  # 'E'
izq, der, arr, abj = explorar_strings_matriz(matriz, pos_inicial)  # (L, R, U, D)

print("Izquierda:", izq)  # "ED"
print("Derecha:  ", der)  # "EF"
print("Arriba:   ", arr)  # "EB"
print("Abajo:    ", abj)  # "EH"


Izquierda: ED
Derecha:   EFY
Arriba:    EB
Abajo:     EHN


---

## 1. Tiempo de ejecución

La función, cuando se invoca con `dir="ROOT"`, hace **cuatro llamadas recursivas independientes** (una por dirección: izquierda, derecha, arriba y abajo).

Analicemos cada caso:

* **Izquierda (L):** desde la columna `j` hasta la columna `0`.
  Se realizan $j$ pasos, cada uno concatenando un carácter.
  → Complejidad: $O(j+1)$.

* **Derecha (R):** desde la columna `j` hasta la columna `m-1`.
  Se realizan $m - j - 1$ pasos.
  → Complejidad: $O(m-j)$.

* **Arriba (U):** desde la fila `i` hasta la fila `0`.
  Se realizan $i$ pasos.
  → Complejidad: $O(i+1)$.

* **Abajo (D):** desde la fila `i` hasta la fila `n-1`.
  Se realizan $n - i - 1$ pasos.
  → Complejidad: $O(n-i)$.

---

### Tiempo total

Si sumamos las cuatro direcciones:

$$
T(n,m,i,j) = O(j+1) + O(m-j) + O(i+1) + O(n-i)
$$

$$
T(n,m,i,j) = O(m) + O(n)
$$

Por lo tanto, **la complejidad temporal total es $O(n+m)$**.
Esto significa que el tiempo de ejecución crece linealmente con el tamaño de la fila y la columna en las que se encuentra la posición inicial.

---

## 2. Espacio en memoria (call stack)

Cada dirección genera una **profundidad de recursión igual a la cantidad de pasos recorridos**:

* Izquierda: profundidad = $j+1$.
* Derecha: profundidad = $m-j$.
* Arriba: profundidad = $i+1$.
* Abajo: profundidad = $n-i$.

El **peor caso** ocurre cuando la posición inicial está en una esquina (por ejemplo, esquina superior izquierda `(0,0)`).
En ese escenario, una dirección puede tener que recorrer toda la fila o columna, con profundidad máxima de:

$$
\max(n, m)
$$

---

### Espacio total

* **Entrada (matriz y posición):** $O(nm)$, ya que la matriz ya está almacenada.
* **Call stack:** $O(\max(n, m))$.
* **Estructuras auxiliares:** solo concatenación de strings, que en Python genera nuevos objetos de tamaño proporcional a la longitud del string construido.

Entonces, la complejidad espacial es:

$$
O(nm) \; (\text{entrada}) + O(\max(n, m)) \; (\text{call stack})
$$

pero si analizamos **solo el espacio adicional de la función**, queda:

$$
O(\max(n, m))
$$

---

✅ **Resumen final**:

* **Tiempo:** $O(n+m)$
* **Espacio auxiliar (call stack):** $O(\max(n,m))$
* **Espacio total:** $O(nm + \max(n,m))$, dominado por la matriz de entrada.

---

# Punto 2

In [13]:
from typing import List, Tuple

def rle_comprimir(s: str) -> Tuple[str, float, bool]:
    n = len(s)
    if n == 0:
        # Definimos tasa = 1.0 para evitar división por cero y expresar "sin cambio"
        return ("", 1.0, False)

    # Helper tail-recursive: la llamada recursiva es SIEMPRE la última operación
    def _rle(i: int, curr: str, count: int, acc: List[str]) -> Tuple[str, float, bool]:
        if i == n:
            # Cerrar la última racha, construir resultado y métricas
            acc.append(curr + str(count))
            comprimido = "".join(acc)
            tasa = (len(comprimido) / n) if n > 0 else 1.0
            return (comprimido, tasa, len(comprimido) < n)
        ch = s[i]
        if ch == curr:
            # Continuamos la racha del mismo carácter
            return _rle(i + 1, curr, count + 1, acc)
        else:
            # Cerramos la racha anterior y empezamos una nueva
            acc.append(curr + str(count))
            return _rle(i + 1, ch, 1, acc)

    # Inicializamos con el primer carácter
    return _rle(i=1, curr=s[0], count=1, acc=[])


# === Ejemplos rápidos ===
# "aaabbc" → "a3b2c1", tasa=6/6=1.0, reduce=False
print(rle_comprimir("aaabbc"))
# "xxxxxxxxxx" (10 x) → "x10", tasa=3/10=0.3, reduce=True
print(rle_comprimir("x" * 10))
# "" → ("", 1.0, False)
print(rle_comprimir(""))
# "ab" → "a1b1", tasa=4/2=2.0, reduce=False
print(rle_comprimir("ab"))


('a3b2c1', 1.0, False)
('x10', 0.3, True)
('', 1.0, False)
('a1b1', 2.0, False)


In [33]:
from typing import List, Tuple

def rle(s: str, i: int = 0, cont: int = 0, compressed: List[str] = [], n: int = None, last_char:str = "") -> Tuple[str, float, bool]:
  if(s == ""):
    return ("",1,False)

  if(n is None):
    n = len(s)
    last_char = s[i]

  if(i == n):
    compressed.append(f"{last_char}{cont}")
    comprimido = "".join(compressed)
    tasa = (len(comprimido) / n) if n > 0 else 1.0
    return (comprimido, tasa, len(comprimido) < n)

  current = s[i]
  if(current == last_char):
    return rle(s, i+1, cont+1, compressed, n, current)
  compressed.append(f"{last_char}{cont}")
  return rle(s, i+1, 1, compressed, n, current)



# === Ejemplos rápidos ===
# "aaabbc" → "a3b2c1", tasa=6/6=1.0, reduce=False
print(rle("aaabbc", compressed = []))
# "xxxxxxxxxx" (10 x) → "x10", tasa=3/10=0.3, reduce=True
print(rle("x" * 10, compressed = []))
# "" → ("", 1.0, False)
print(rle("", compressed = []))
# "ab" → "a1b1", tasa=4/2=2.0, reduce=False
print(rle("ab", compressed = []))

('a3b2c1', 1.0, False)
('x10', 0.3, True)
('', 1, False)
('a1b1', 2.0, False)


### Complejidad

* **Tiempo:** $O(n)$. Cada carácter del string se procesa exactamente una vez.
* **Espacio auxiliar (call stack):** $O(n)$ en el peor caso (un único bloque, p. ej. `"aaaaa..."`), por la profundidad recursiva.
  Además, el acumulador `acc` almacena como mucho tantas piezas como cambios de racha (acotado por $O(n)$).


# Punto 3

In [34]:
from typing import Literal, Tuple

Mode = Literal["ROOT", "STEP"]

def comparar_vocales_consonantes(s: str, i: int = 0, mode: Mode = "ROOT") -> str | Tuple[int, int]:
    # Conjunto de vocales en español (incluye tildes y diéresis) en minúsculas:
    vocales = {"a", "e", "i", "o", "u", "á", "é", "í", "ó", "ú", "ü"}

    if mode == "ROOT":
        v, c = comparar_vocales_consonantes(s, 0, "STEP")  # obtiene conteos
        # El enunciado garantiza que no hay empates
        return "más vocales" if v > c else "más consonantes"

    # mode == "STEP": devolvemos conteos acumulados (v, c)
    if i == len(s):
        return (0, 0)

    # Llamada recursiva primero (NO de cola: trabajo después)
    v_rec, c_rec = comparar_vocales_consonantes(s, i + 1, "STEP")

    ch = s[i].lower()
    if ch.isalpha():
        if ch in vocales:
            return (v_rec + 1, c_rec)
        else:
            return (v_rec, c_rec + 1)
    else:
        # Caracter no alfabético: se ignora para el conteo
        return (v_rec, c_rec)


In [35]:
print(comparar_vocales_consonantes("aymuchachos"))          # "más consonantes"
print(comparar_vocales_consonantes("nocanceleeeeeeeeeeeeeeemos"))  # "más vocales"

más consonantes
más vocales


In [62]:
from typing import Literal, Tuple

Mode = Literal["ROOT", "STEP"]

def comparar_vocales_consonantes(s: str, i: int = 0, cv: int = 0, cc: int = 0) -> str | Tuple[int, int]:
    # Conjunto de vocales en español (incluye tildes y diéresis) en minúsculas:
    vocales = {"a", "e", "i", "o", "u", "á", "é", "í", "ó", "ú", "ü"}
    # mode == "STEP": devolvemos conteos acumulados (v, c)
    if i == len(s):
        return cv, cc

    ch = s[i].lower()
    if(ch in vocales):
      cv, cc = comparar_vocales_consonantes(s, i+1, cv+1, cc)
    else:
      cv, cc = comparar_vocales_consonantes(s, i+1, cv, cc+1)

    if(i == 0):
      return "más vocales" if cv > cc else "más consonantes"
    return cv, cc

In [66]:
print(comparar_vocales_consonantes("aymuchachos"))          # "más consonantes"
print(comparar_vocales_consonantes("nocanceleeeeeeeeeeeeeeemos"))  # "más vocales"
print(comparar_vocales_consonantes("xaaaaaaaaa"))  # "más vocales"

más consonantes
más vocales
más vocales
